# Resumidor de commits: diff → mensaje en lenguaje natural

**Proyecto:** Mini-GitHub — commit-summarizer-ms · **Fase:** Prompt 1 (Machine Learning)

**Dataset:** corpus limpio de NNGen (Liu et al., ASE 2018), derivado del corpus de **Jiang, Armaly & McMillan (ASE 2017)** — el paper citado en la bibliografia del proyecto. ~27k pares diff→mensaje de los 1,000 proyectos Java mas populares de GitHub. Fuente: https://github.com/Tbabm/nngen

**Modelo:** *sequence-to-sequence* entrenado **desde cero** (cumple el limite del proyecto: sin modelos pre-entrenados): encoder GRU bidireccional + decoder GRU con **atencion de Luong**. Es la arquitectura de la linea de investigacion citada (NMT aplicada a commits).

**Metricas:** BLEU-4 y ROUGE-L, como exige el documento del proyecto.

> ⚠️ En Colab: **Entorno de ejecucion → Cambiar tipo de entorno → GPU (T4)** antes de ejecutar. El entrenamiento toma ~15-25 min.

In [ ]:
# ── 1. Dependencias, semillas y GPU ─────────────────────────────────────────
!pip -q install sacrebleu rouge-score

import json, os, re, random, zipfile, glob, math, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

assert torch.cuda.is_available(), 'Activa la GPU: Entorno de ejecucion -> Cambiar tipo de entorno -> T4 GPU'
DEVICE = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

In [ ]:
# ── 2. Descarga del dataset e inspeccion ────────────────────────────────────
!git clone --depth 1 https://github.com/Tbabm/nngen.git 2>/dev/null || echo 'ya clonado'

for raiz, dirs, archivos in os.walk('nngen'):
    if '.git' in raiz:
        continue
    print(raiz + '/')
    for a in archivos:
        ruta = os.path.join(raiz, a)
        print(f'   {a}  ({os.path.getsize(ruta)/1e6:.1f} MB)')

In [ ]:
# ── 3. Carga de pares diff→mensaje ──────────────────────────────────────────
# El corpus viene pre-tokenizado, un ejemplo por linea, en archivos paralelos
# *.diff / *.msg para train/valid/test. Verificamos el paralelismo (mismo
# numero de lineas) antes de usarlo.
def cargar(split):
    diff_path = next(iter(glob.glob(f'nngen/data/**/*{split}*.diff', recursive=True)))
    msg_path = next(iter(glob.glob(f'nngen/data/**/*{split}*.msg', recursive=True)))
    with open(diff_path, encoding='utf-8', errors='replace') as f:
        diffs = [l.strip() for l in f]
    with open(msg_path, encoding='utf-8', errors='replace') as f:
        msgs = [l.strip() for l in f]
    assert len(diffs) == len(msgs), f'{split}: archivos no paralelos'
    print(f'{split}: {len(diffs)} pares  (ej: "{msgs[0][:70]}...")')
    return diffs, msgs

train_diffs, train_msgs = cargar('train')
val_diffs, val_msgs = cargar('valid')
test_diffs, test_msgs = cargar('test')

# Estadisticas de longitud para justificar el truncado
lens_d = [len(d.split()) for d in train_diffs]
lens_m = [len(m.split()) for m in train_msgs]
print(f'diff tokens: mediana={int(np.median(lens_d))} p90={int(np.percentile(lens_d,90))}')
print(f'msg tokens:  mediana={int(np.median(lens_m))} p90={int(np.percentile(lens_m,90))}')

In [ ]:
# ── 4. Vocabularios y tensorizacion ─────────────────────────────────────────
# Truncamos el diff a 100 tokens y el mensaje a 25 (igual que Jiang et al.):
# cubre a la mayoria (ver percentiles arriba) y acota memoria/computo.
MAX_DIFF, MAX_MSG = 100, 25
PAD, SOS, EOS, UNK = 0, 1, 2, 3
ESPECIALES = ['<pad>', '<sos>', '<eos>', '<unk>']

def construir_vocab(textos, max_tam, min_frec=2):
    from collections import Counter
    c = Counter(tok for t in textos for tok in t.split())
    palabras = [w for w, n in c.most_common(max_tam - len(ESPECIALES)) if n >= min_frec]
    itos = ESPECIALES + palabras
    stoi = {w: i for i, w in enumerate(itos)}
    return stoi, itos

stoi_d, itos_d = construir_vocab(train_diffs, 50_000)
stoi_m, itos_m = construir_vocab(train_msgs, 20_000)
print(f'vocab diff: {len(itos_d)} | vocab msg: {len(itos_m)}')

def codificar(texto, stoi, max_len, agregar_fin=False):
    ids = [stoi.get(t, UNK) for t in texto.split()[:max_len]]
    if agregar_fin:
        ids.append(EOS)
    return ids

class ParesDataset(Dataset):
    def __init__(self, diffs, msgs):
        self.x = [codificar(d, stoi_d, MAX_DIFF) for d in diffs]
        self.y = [codificar(m, stoi_m, MAX_MSG, agregar_fin=True) for m in msgs]
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

def colacionar(lote):
    xs, ys = zip(*lote)
    lx = max(len(x) for x in xs); ly = max(len(y) for y in ys)
    X = torch.full((len(xs), lx), PAD, dtype=torch.long)
    Y = torch.full((len(ys), ly), PAD, dtype=torch.long)
    for i, (x, y) in enumerate(zip(xs, ys)):
        X[i, :len(x)] = torch.tensor(x); Y[i, :len(y)] = torch.tensor(y)
    return X, Y

dl_train = DataLoader(ParesDataset(train_diffs, train_msgs), batch_size=64, shuffle=True, collate_fn=colacionar)
dl_val = DataLoader(ParesDataset(val_diffs, val_msgs), batch_size=64, shuffle=False, collate_fn=colacionar)
print(f'{len(dl_train)} lotes de entrenamiento')

In [ ]:
# ── 5. Modelo: encoder biGRU + decoder GRU con atencion de Luong ────────────
EMB, HID = 256, 256

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(itos_d), EMB, padding_idx=PAD)
        self.gru = nn.GRU(EMB, HID, batch_first=True, bidirectional=True)
        self.proy = nn.Linear(HID * 2, HID)
    def forward(self, x):
        salidas, h = self.gru(self.emb(x))              # salidas: B,L,2H
        h0 = torch.tanh(self.proy(torch.cat([h[0], h[1]], dim=1)))  # B,H
        return salidas, h0.unsqueeze(0)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(itos_m), EMB, padding_idx=PAD)
        self.gru = nn.GRU(EMB, HID, batch_first=True)
        self.atn_proj = nn.Linear(HID, HID * 2)          # puntaje Luong general
        self.salida = nn.Linear(HID * 3, len(itos_m))
    def forward(self, y_prev, h, enc_out, mascara):
        e = self.emb(y_prev)                             # B,1,E
        o, h = self.gru(e, h)                            # o: B,1,H
        puntajes = torch.bmm(self.atn_proj(o), enc_out.transpose(1, 2))  # B,1,L
        puntajes = puntajes.masked_fill(mascara.unsqueeze(1), -1e9)
        pesos = torch.softmax(puntajes, dim=-1)
        contexto = torch.bmm(pesos, enc_out)             # B,1,2H
        logits = self.salida(torch.cat([o, contexto], dim=-1))  # B,1,V
        return logits, h

encoder, decoder = Encoder().to(DEVICE), Decoder().to(DEVICE)
params = sum(p.numel() for p in list(encoder.parameters()) + list(decoder.parameters()))
print(f'Parametros: {params/1e6:.1f}M (entrenado desde cero, sin pre-entrenamiento)')

In [ ]:
# ── 6. Entrenamiento con teacher forcing + parada temprana ──────────────────
# Hasta 25 epocas, deteniendo si la validacion no mejora en 3 seguidas:
# aprovecha el computo disponible sin sobreajustar.
EPOCAS, PACIENCIA = 25, 3
opt = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)
perdida_fn = nn.CrossEntropyLoss(ignore_index=PAD)

def paso(X, Y, entrenar=True):
    X, Y = X.to(DEVICE), Y.to(DEVICE)
    enc_out, h = encoder(X)
    mascara = (X == PAD)
    B, Ly = Y.shape
    entrada = torch.full((B, 1), SOS, dtype=torch.long, device=DEVICE)
    perdida = 0.0
    for t in range(Ly):
        logits, h = decoder(entrada, h, enc_out, mascara)
        perdida = perdida + perdida_fn(logits.squeeze(1), Y[:, t])
        entrada = Y[:, t:t+1]                            # teacher forcing
    perdida = perdida / Ly
    if entrenar:
        opt.zero_grad(); perdida.backward()
        nn.utils.clip_grad_norm_(list(encoder.parameters()) + list(decoder.parameters()), 5.0)
        opt.step()
    return perdida.item()

mejor_val, sin_mejora = float('inf'), 0
for ep in range(1, EPOCAS + 1):
    t0 = time.time()
    encoder.train(); decoder.train()
    perdidas = [paso(X, Y) for X, Y in dl_train]
    encoder.eval(); decoder.eval()
    with torch.no_grad():
        val = np.mean([paso(X, Y, entrenar=False) for X, Y in dl_val])
    marca = ''
    if val < mejor_val:
        mejor_val, sin_mejora = val, 0
        torch.save({'encoder': encoder.state_dict(), 'decoder': decoder.state_dict()}, 'mejor_modelo.pt')
        marca = ' *guardado*'
    else:
        sin_mejora += 1
    print(f'epoca {ep:02d} | train {np.mean(perdidas):.3f} | val {val:.3f} | {time.time()-t0:.0f}s{marca}')
    if sin_mejora >= PACIENCIA:
        print(f'Parada temprana: {PACIENCIA} epocas sin mejorar validacion.')
        break

In [ ]:
# ── 7. Inferencia con beam search y ejemplos cualitativos ───────────────────
# Beam search (ancho 5) en lugar de greedy: explora varias hipotesis y elige la
# de mayor probabilidad normalizada por longitud. Mejora tipica de +1 a +3 BLEU.
ckpt = torch.load('mejor_modelo.pt')
encoder.load_state_dict(ckpt['encoder']); decoder.load_state_dict(ckpt['decoder'])
encoder.eval(); decoder.eval()

@torch.no_grad()
def resumir(diff_texto, beam=5, max_len=MAX_MSG):
    ids = codificar(diff_texto, stoi_d, MAX_DIFF) or [UNK]
    X = torch.tensor([ids], device=DEVICE)
    enc_out, h = encoder(X)
    mascara = (X == PAD)
    haces = [(0.0, [], h, False)]  # (log_prob, tokens, estado, terminado)
    for _ in range(max_len):
        candidatos = []
        for lp, toks, hh, fin in haces:
            if fin:
                candidatos.append((lp, toks, hh, True))
                continue
            ultimo = SOS if not toks else toks[-1]
            entrada = torch.full((1, 1), ultimo, dtype=torch.long, device=DEVICE)
            logits, h2 = decoder(entrada, hh, enc_out, mascara)
            logprobs = torch.log_softmax(logits.view(-1), dim=-1)
            top_lp, top_ix = logprobs.topk(beam)
            for k in range(beam):
                tok = top_ix[k].item()
                candidatos.append((lp + top_lp[k].item(), toks + [tok], h2, tok == EOS))
        haces = sorted(candidatos, key=lambda c: c[0] / max(len(c[1]), 1), reverse=True)[:beam]
        if all(c[3] for c in haces):
            break
    mejores_tokens = haces[0][1]
    return ' '.join(itos_m[t] for t in mejores_tokens if t not in (EOS, SOS, PAD))

for i in random.sample(range(len(test_diffs)), 5):
    print('REAL:     ', test_msgs[i][:90])
    print('GENERADO: ', resumir(test_diffs[i])[:90])
    print('─' * 70)

In [ ]:
# ── 8. Evaluacion: BLEU-4 y ROUGE-L sobre test ──────────────────────────────
# PROTOCOLO: el corpus NNGen ya viene tokenizado (espacios alrededor de la
# puntuacion). La literatura (Jiang 2017, Liu 2018) calcula BLEU sobre ese
# texto tokenizado tal cual; por eso usamos tokenize='none' — re-tokenizar
# con sacrebleu deprime el puntaje y hace la comparacion injusta.
import sacrebleu
from rouge_score import rouge_scorer

print('Generando resumenes del test con beam search (2-4 min)...')
generados = [resumir(d) for d in test_diffs]

bleu = sacrebleu.corpus_bleu(generados, [test_msgs], tokenize='none', force=True)
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rougeL = np.mean([scorer.score(r, g)['rougeL'].fmeasure for r, g in zip(test_msgs, generados)])

print(f'BLEU-4 (tokenizado, protocolo de la literatura): {bleu.score:.2f}')
print(f'ROUGE-L: {rougeL:.4f}')
# Referencia con el MISMO dataset y protocolo (Liu et al. 2018, tabla 1):
#   NMT (seq2seq atencional grande): BLEU ~16.4
#   NNGen (recuperacion, sin red):   BLEU ~38.5
# Nuestro seq2seq compacto (~15M parametros, 1 sesion de Colab) debe quedar
# por debajo del NMT grande; documentar la brecha y sus causas (tamano del
# modelo, computo, sin ensembles) es parte de la evaluacion honesta.

In [ ]:
# ── 9. Exportacion de artefactos ────────────────────────────────────────────
from google.colab import files
os.makedirs('artefactos', exist_ok=True)

torch.save({
    'encoder': encoder.state_dict(),
    'decoder': decoder.state_dict(),
    'hiperparametros': {'EMB': EMB, 'HID': HID, 'MAX_DIFF': MAX_DIFF, 'MAX_MSG': MAX_MSG},
}, 'artefactos/modelo_resumidor.pt')

with open('artefactos/vocab_diff.json', 'w') as f:
    json.dump(itos_d, f)
with open('artefactos/vocab_msg.json', 'w') as f:
    json.dump(itos_m, f)

metadatos = {
    'arquitectura': 'seq2seq GRU bidireccional + atencion Luong (desde cero)',
    'bleu4_test': round(float(bleu.score), 2),
    'rougeL_test': round(float(rougeL), 4),
    'torch_version': torch.__version__,
    'dataset': 'github:Tbabm/nngen (corpus de Jiang et al. 2017, limpiado por Liu et al. 2018)',
    'seed': SEED,
    'contrato_inferencia': {
        'entrada': {'diff': 'string (salida de git diff, texto plano)'},
        'preproceso': 'tokenizacion por espacios, truncado a 100 tokens',
        'salida': {'resumen': 'string en ingles, max 25 tokens'},
    },
}
with open('artefactos/metricas_resumidor.json', 'w') as f:
    json.dump(metadatos, f, indent=2)

with zipfile.ZipFile('artefactos_resumidor.zip', 'w') as z:
    for fn in os.listdir('artefactos'):
        z.write(f'artefactos/{fn}', fn)

files.download('artefactos_resumidor.zip')
print('Listo. Guarda el zip en Github-commit-summarizer-ms/training/artefactos/')